In [ ]:
# 1. 載入必要的函式庫 (Import Libraries)
%matplotlib inline
import matplotlib.pyplot as plt
import mdtraj as md
from contact_map import ContactFrequency # 確保您已安裝 contact-map 套件
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import os

# --- 檔案路徑 (已使用您提供的路徑) ---
PDB_FILE = ".pdb"      # PDB 拓撲檔案
DCD_FILE = ".dcd"    # DCD 軌跡檔案
# ---------------------------------

# 2. 載入 PDB 和 DCD 檔案 (Load PDB and DCD Files using mdtraj)
try:
    # 加入 stride=10，每 10 幀取 1 幀進行分析
    traj = md.load(DCD_FILE, top=PDB_FILE, stride=100)
    topology = traj.topology
    print("Trajectory loaded successfully with stride=10.")
    print(traj)
except Exception as e:
    print(f"錯誤：無法載入檔案。請確認 PDB 和 DCD 路徑是否正確。")
    print(f"Error details: {e}")
    exit()

# 3. 選擇所有 DNA 殘基 (Select All DNA Residues)
# 定義標準 DNA 殘基名稱的集合 (set)，並加入單字母版本以增加通用性
DNA_RESIDUE_NAMES = {'DA', 'DT', 'DC', 'DG', 'A', 'T', 'C', 'G'} 
dna_residues = [
    residue for residue in topology.residues 
    if residue.name in DNA_RESIDUE_NAMES
]

# 獲取這些殘基中所有原子的索引
dna_atom_indices = [atom.index for residue in dna_residues for atom in residue.atoms]
# 獲取殘基的標籤 (名稱 + PDB 編號) 用於繪圖
dna_labels = [f"{res.name}{res.resSeq}" for res in dna_residues]

# 打印檢查
if not dna_residues:
    print("警告：找不到任何標準的 DNA 殘基。請檢查 PDB 檔案中的殘基名稱。")
else:
    print(f"Found {len(dna_residues)} DNA residues.")
    print(f"Residue labels: {', '.join(dna_labels[:15])}...") 


# 4. 計算接觸頻率 (Calculate Contact Frequency)
# 計算 DNA 對自身的接觸
trajectory_contacts = ContactFrequency(
    traj, 
    cutoff=1.2, # mdtraj 單位為 nm
    haystack=dna_atom_indices, 
    query=dna_atom_indices
)


# 5. 繪製接觸圖 (Plot Contact Map)

# --- 修改點：定義並使用白色到紅色的 cmap ---
red_cmap = LinearSegmentedColormap.from_list("RedOnly", ["white", "red"])

fig, ax = trajectory_contacts.residue_contacts.plot(
    cmap=red_cmap, vmin=0, vmax=1
)
# ---------------------------------------------

# --- 客製化繪圖細節 (Customizing the Plot) ---
fig.set_size_inches(16, 14) 
fig.set_dpi(300)

# 修正 x 軸和 y 軸範圍
ax.set_xlim(0, len(dna_labels))
ax.set_ylim(0, len(dna_labels))

# 添加水平和垂直虛線
for i in range(len(dna_labels)):
    ax.hlines(i, 0, len(dna_labels), colors="gray", linestyles="dashed", lw=0.5)
    ax.vlines(i, 0, len(dna_labels), colors="gray", linestyles="dashed", lw=0.5)

# 調整 colorbar 標籤大小
cax = fig.axes[1]
cax.yaxis.set_tick_params(labelsize=14)
cax.set_ylabel("Contact Frequency", fontsize=16)

# 設定 x 和 y 軸標籤
plt.xlabel("DNA Residue", fontsize=24)
plt.ylabel("DNA Residue", fontsize=24)

# 設定 x, y 軸刻度標籤
plt.xticks(
    ticks=np.arange(len(dna_labels)), 
    labels=dna_labels, 
    fontsize=16, 
    rotation=90
)
plt.yticks(
    ticks=np.arange(len(dna_labels)), 
    labels=dna_labels, 
    fontsize=16
)

# 自動調整佈局以防止截斷
plt.tight_layout(pad=1.5)

# 顯示圖表，不儲存檔案
plt.show()